In [1]:
import os
from cocoa_fisher import FisherMeta, FisherCase

REAL_MVS = [i_mv for i_mv in range(1, 13) if i_mv not in [8, 9]]  # [8, 9, 11, 12]
FOURIER_MVS = [i_mv for i_mv in range(1, 12) if i_mv not in [8]]  # [5, 8, 11]

submit_jobs = False  # 6/20/2025, 7/23/2025
if submit_jobs:
    with open("cocoa_dcsolver.sh", "r") as f:
        LINES = f.readlines()

In [2]:
if submit_jobs:
    space, n_iter = "real", 10  # "real" or "fourier"

    for i_mv in [1]:  # vars()[f"{space.upper()}_MVS"]:
        lines = LINES.copy()
        for i, line in enumerate(lines):
            if "--job-name" in line:
                lines[i] = f"#SBATCH --job-name=dc1_{space}_v{i_mv}\n"
            if "--output" in line:
                lines[i] = f"#SBATCH --output=dc1_{space}_v{i_mv}.out\n"
            if "python" in line:
                lines[i] = f"python cocoa_dcsolver.py {space} {i_mv} {n_iter}\n"

        with open(f"cocoa_dcsolver_{space}_v{i_mv}.sh", "w") as f:
            f.writelines(lines)
        os.system(f"sbatch cocoa_dcsolver_{space}_v{i_mv}.sh")
        os.system(f"rm cocoa_dcsolver_{space}_v{i_mv}.sh")

In [3]:
# roman_cpip_data_challenge/results.tex
template = dict(
    omegam=r"$\Omega_m$ \\", sigma8=r"$\sigma_8$ \\",
    ns=r"$n_s$ \\", omegab=r"$\Omega_b$ \\", h0=r"$h_0$ \\",
    **{"roman_B1_"+str(i+1): rf"$b^{i+1}$ \\" for i in range(8)},
    **{"roman_DZ_S"+str(i+1): rf"$\Delta_{{z}}^{i+1}$ \\" for i in range(8)},
    **{"roman_M"+str(i+1): rf"$m_{i+1}$ \\" for i in range(8)},
    roman_A1_1=r"$A_{\rm IA}$ \\", roman_A1_2=r"$\eta_{\rm IA}$ \\"
)

section = dict(
    omegam="Cosmology", roman_B1_1="Galaxy bias",
    roman_DZ_S1="Photo-z", roman_M1="Shear calibration", roman_A1_1="IA"
)

In [4]:
def report_results(space):
    results_ML = template.copy()
    results_MAP = template.copy()
    if space == "real": exclude_mvs = [11, 12]
    elif space == "fourier": exclude_mvs = [5, 11]

    for i_mv in globals()[f"{space.upper()}_MVS"]:
        print(f"Processing {space} i_mv = {i_mv}", flush=True)
        if i_mv in exclude_mvs:
            for param, line in results_ML.items():
                results_ML[param] = line[:-3] + " & ---" + line[-3:]
            for param, line in results_MAP.items():
                results_MAP[param] = line[:-3] + " & ---" + line[-3:]
            continue

        for posterior in [False, True]:
            FisherMeta.POSTERIOR = posterior
            base = FisherCase(FisherMeta(space, i_mv))
            base.compute_values_and_errors()

            results = [results_ML, results_MAP][posterior]
            for param, line in results.items():
                idx = base.allparams.index(param)
                results[param] = line[:-3] + " & $" + f"{base.values_ML[idx]:.4f}" + r" \pm " +\
                    f"{(base.errors_ML if not posterior else base.errors_MAP)[idx]:.4f}" + "$" + line[-3:]

    for results in [results_MAP, results_ML]:
        print()
        for param, line in results.items():
            if (title := section.get(param)) is not None:
                print(r"\hline" + "\n" + r"\multicolumn{10}{l}{\textbf{" + title + r"}} \\")
            print(line)

In [5]:
# report_results("real")

In [6]:
# report_results("fourier")